# **Loading Packages**
_____

In [19]:
import pandas as pd
import numpy as np

import requests
import time
from io import StringIO
import os


from fredapi import Fred
import yfinance as yf

# **Calling Data**
______

## **FRED**

In [ ]:
with open("API_key.txt", "r") as file:
    api_key = file.read().strip()
fred = Fred(api_key=api_key)

exchange_rate = fred.get_series('RBGBBIS').resample('QS').first()
oil_price = fred.get_series('DCOILBRENTEU').resample('QS').first()
bond_yield = fred.get_series('IRLTLT01GBM156N').resample('QS').first()
UK_GDP = fred.get_series('NGDPRSAXDCGBQ')

Fred_Data = pd.DataFrame({
    'ExchangeRate': exchange_rate,
    'OilPrice': oil_price,
    'BondYield': bond_yield,
    'rGDP': UK_GDP
})

Fred_Data = Fred_Data[Fred_Data.index > '1993-10-01']

## **Office of National Stats**

## **Yahoo Finance**

In [47]:
ftse = yf.Ticker("^FTSE")

ftse_data = ftse.history(
    start="1984-01-01",  
    end=pd.Timestamp.today().strftime('%Y-%m-%d'),
    interval="3mo"  
)

Thoughts on alternative resampling options:
- ftse_quarterly = ftse_data.resample('QE').mean()  
- ftse_quarterly = ftse_data.resample('QE').first()  
- ftse_quarterly = ftse_data['Close'].resample('QE').last()  

## **OECD**

In [53]:
def fetch_oecd(url, name):
    """Fetch OECD data and return cleaned DataFrame"""
    if 'format=' not in url:
        url += '&format=csvfilewithlabels'
    
    headers = {
        'Accept': 'application/vnd.sdmx.data+csv; charset=utf-8; labels=both',
        'Accept-Encoding': 'gzip, deflate'
    }
    
    time.sleep(1)
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        df = pd.read_csv(StringIO(response.text))
        
        # Clean: keep only essential columns
        df_clean = df[['TIME_PERIOD', 'OBS_VALUE']].copy()
        df_clean.columns = ['date', name]
        df_clean[name] = pd.to_numeric(df_clean[name], errors='coerce')
        return df_clean
    else:
        print(f"✗ {name}: Error {response.status_code}")
        return None

# ======================================================================
# Update URLs here
# ======================================================================
queries = {
    "house_prices": "https://sdmx.oecd.org/public/rest/data/OECD.ECO.MPD,DSD_AN_HOUSE_PRICES@DF_HOUSE_PRICES,/GBR.Q.RHP.?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions",
    "cpi": "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_G20_PRICES@DF_G20_PRICES,/GBR.Q...PA...?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions",
    "unemployment": "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_LFS@DF_IALFS_UNE_M,/GBR..._Z.Y._T.Y_GE15..Q?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions"
}

dfs = [fetch_oecd(url, name) for name, url in queries.items()]
OECDdata = pd.concat([df.set_index('date') for df in dfs if df is not None], axis=1).reset_index()



# **Creating Dataset**
___

In [54]:
# Convert date column to datetime and set as index
OECDdata['date'] = pd.to_datetime(OECDdata['date']).dt.tz_localize(None)
OECDdata.set_index('date', inplace=True)

# Convert FRED and FTSE indexes to datetime and remove timezone
Fred_Data.index = pd.to_datetime(Fred_Data.index).tz_localize(None)
ftse_data.index = pd.to_datetime(ftse_data.index).tz_localize(None)

# Merge
merged = OECDdata.join([Fred_Data, ftse_data], how='outer')

# Check for duplicates
print(f"Duplicates: {merged.index.duplicated().sum()}")

# Remove duplicates if they exist
if merged.index.duplicated().any():
    merged = merged[~merged.index.duplicated(keep='first')]

Duplicates: 0


/var/folders/ts/x847bxb170n3msnnpmdpjll00000gn/T/ipykernel_73236/1763363256.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  OECDdata['date'] = pd.to_datetime(OECDdata['date']).dt.tz_localize(None)


In [55]:
merged = merged.drop(['High','Close','Low','Volume','Dividends','Stock Splits'],axis=1)
merged = merged[merged.index > '1993-12-31']

In [56]:
merged

,house_prices,cpi,unemployment,ExchangeRate,OilPrice,BondYield,rGDP,Open
1994-01-01,40.501848,2.466667,9.9,114.76,13.43,6.3794,391026.0,3427.199951
1994-04-01,40.642671,2.000000,9.7,112.78,14.33,7.8401,394588.0,3060.899902
1994-07-01,40.633868,1.700000,9.4,111.17,17.65,8.5330,398566.0,2914.100098
1994-10-01,40.629881,1.766667,9.0,112.48,16.85,8.7797,400279.0,3024.300049
1995-01-01,40.077473,2.466667,8.9,111.13,15.88,8.7349,402390.0,3062.600098
...,...,...,...,...,...,...,...,...
2025-01-01,109.901506,2.800000,4.5,109.95,76.14,4.6627,703435.0,8173.000000
2025-04-01,108.058076,3.500000,4.7,112.45,77.78,4.5762,704973.0,8582.799805
2025-07-01,108.078358,3.800000,5.0,112.29,67.63,4.5924,705603.0,8761.000000
2025-10-01,NaN,3.400000,NaN,110.56,66.67,4.5721,NaN,9350.500000


In [ ]:
#Be sure to change to your own internal directory
#os.chdir('/Users/dannyhogan/Desktop/ST-498')
#merged.to_csv("Data/MacroVariables.csv")